<a href="https://colab.research.google.com/github/Kaiking28/ECON3916-Statistical-Machine-Learning/blob/main/class9/lab9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

# Load your dataset here (ensure lalonde.csv is uploaded to Colab or linked)
df = pd.read_csv('lalonde_obs.csv')

# Naive Comparison
naive_diff = df[df.treat == 1]['re78'].mean() - df[df.treat == 0]['re78'].mean()
print(f"Naive Difference in Means: ${naive_diff:,.2f}")
# Expected Result: -$635.03

# Define covariates
X = df[['age', 'educ', 'black', 'hisp', 'married', 'nodegr', 're74', 're75']]
y = df['treat']

# Fit Propensity Model
logit = LogisticRegression(solver='liblinear')
logit.fit(X, y)

# Generate Scores
df['pscore'] = logit.predict_proba(X)[:, 1]


from sklearn.neighbors import NearestNeighbors

# Separate groups
treated = df[df.treat == 1].reset_index(drop=True)
control = df[df.treat == 0].reset_index(drop=True)

# Fit NN on Control scores
nbrs = NearestNeighbors(n_neighbors=1, algorithm='ball_tree').fit(control[['pscore']])

# Find matches for Treated scores
distances, indices = nbrs.kneighbors(treated[['pscore']])
matched_control = control.iloc[indices.flatten()].reset_index(drop=True)

# Construct Matched DataFrame
matched_df = pd.concat([treated, matched_control]).reset_index(drop=True)


from scipy import stats

# T-test on raw data
diff = treated['re78'].mean() - control['re78'].mean()
t_stat, p_val = stats.ttest_ind(treated['re78'], control['re78'])

print(f"Raw Effect (Difference): ${diff:,.2f}")
print(f"P-value: {p_val:.4f}")


# Isolate the matched outcomes
matched_treated = matched_df[matched_df.treat == 1]['re78']
matched_control = matched_df[matched_df.treat == 0]['re78']

# Estimate the causal effect (T-test on matched data)
matched_diff = matched_treated.mean() - matched_control.mean()
t_stat, p_val = stats.ttest_ind(matched_treated, matched_control)

print(f"Recovered Effect (Matched Difference): ${matched_diff:,.2f}")
print(f"P-value: {p_val:.4f}")


FileNotFoundError: [Errno 2] No such file or directory: 'lalonde_obs.csv'